# TripMind review sentiment (TF-IDF + logistic regression)

Scores hotel reviews as positive, neutral or negative so TripMind can summarise what guests
say about a stay (overall and per aspect: location, staff, cleanliness, ...).

**Data:** *Trip Advisor Hotel Reviews* (about 20,000 TripAdvisor reviews with a 1-5 rating),
Kaggle dataset `andrewmvd/trip-advisor-hotel-reviews`. Check the licence on the dataset page
before use; a Hugging Face mirror of the same data lists **CC BY-NC 4.0** (non-commercial).
Labels: 1-2 stars negative, 3 neutral, 4-5 positive.

**Run on Kaggle:** new notebook, *Add data*, search "Trip Advisor Hotel Reviews", *Run all*.
CPU is enough.

**Hand back:** download `model.json` and `model_card.json` from `/kaggle/working` into
`models/review_sentiment/artifacts/`. The model is plain JSON (no pickle). The notebook stops
with an error unless the model beats the word-list rules TripMind uses today.

In [ ]:
import glob
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

frames = [pd.read_csv(f) for f in glob.glob("/kaggle/input/**/*.csv", recursive=True)]
df = next(f for f in frames if {"review", "rating"} <= {c.lower() for c in f.columns})
df.columns = [c.lower() for c in df.columns]
df = df[["review", "rating"]].dropna()
df = df[df.rating.between(1, 5)]
df["label"] = np.select([df.rating <= 2, df.rating == 3], ["negative", "neutral"], "positive")
print(len(df), df.label.value_counts().to_dict())

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=7, stratify=df.label)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=30_000, sublinear_tf=True)
model = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced")
model.fit(vectorizer.fit_transform(train.review), train.label)
predicted = model.predict(vectorizer.transform(test.review))
print(classification_report(test.label, predicted, digits=3))

**Baseline:** the word-list rules TripMind serves without a model (copied from `backend/app/modules/ml/sentiment.py`).

In [ ]:
import re
from dataclasses import dataclass
from typing import Literal

Label = Literal["positive", "neutral", "negative"]

POSITIVE = frozenset(
    (
        "amazing",
        "awesome",
        "beautiful",
        "best",
        "brilliant",
        "clean",
        "comfortable",
        "comfy",
        "convenient",
        "cosy",
        "cozy",
        "delicious",
        "delightful",
        "excellent",
        "fabulous",
        "fantastic",
        "fast",
        "friendly",
        "gorgeous",
        "great",
        "helpful",
        "hospitable",
        "ideal",
        "immaculate",
        "impressive",
        "love",
        "loved",
        "lovely",
        "nice",
        "outstanding",
        "peaceful",
        "perfect",
        "pleasant",
        "polite",
        "quiet",
        "recommend",
        "recommended",
        "relaxing",
        "reliable",
        "roomy",
        "spacious",
        "spotless",
        "stunning",
        "superb",
        "tasty",
        "value",
        "welcoming",
        "wonderful",
        "worth",
    )
)

NEGATIVE = frozenset(
    (
        "awful",
        "bad",
        "bland",
        "broken",
        "cold",
        "cramped",
        "dated",
        "dirty",
        "disappointed",
        "disappointing",
        "disgusting",
        "dropping",
        "expensive",
        "filthy",
        "horrible",
        "noisy",
        "old",
        "overpriced",
        "poor",
        "rude",
        "shabby",
        "slow",
        "small",
        "smelly",
        "stained",
        "terrible",
        "tired",
        "unclean",
        "unfriendly",
        "unhelpful",
        "unpleasant",
        "worst",
    )
)

NEGATORS = frozenset(
    (
        "not",
        "no",
        "never",
        "nobody",
        "hardly",
        "barely",
        "didn't",
        "don't",
        "wasn't",
        "weren't",
        "isn't",
    )
)

_WORDS = re.compile(r"[a-z']+")

@dataclass(frozen=True)
class Sentiment:
    label: Label
    score: float  # -1 (negative) .. 1 (positive)

def _rules(text: str) -> Sentiment:
    words = _WORDS.findall(text.lower())
    total = 0.0
    hits = 0
    for i, word in enumerate(words):
        polarity = 1 if word in POSITIVE else -1 if word in NEGATIVE else 0
        if not polarity:
            continue
        if any(w in NEGATORS for w in words[max(0, i - 3) : i]):
            polarity = -polarity
        total += polarity
        hits += 1
    score = total / hits if hits else 0.0
    label: Label = "positive" if score > 0.2 else "negative" if score < -0.2 else "neutral"
    return Sentiment(label, round(score, 3))


baseline = [_rules(text).label for text in test.review]
metrics = {
    "macro_f1": f1_score(test.label, predicted, average="macro"),
    "accuracy": accuracy_score(test.label, predicted),
    "baseline_macro_f1": f1_score(test.label, baseline, average="macro"),
    "baseline_accuracy": accuracy_score(test.label, baseline),
    "test_rows": float(len(test)),
}
print(metrics)
assert metrics["macro_f1"] > metrics["baseline_macro_f1"], "model does not beat the rules"

In [ ]:
from __future__ import annotations

import datetime
import json
from pathlib import Path
from typing import Any

FORMAT = "tfidf-logreg-v1"


def export_tfidf_logreg(
    vectorizer: Any,
    model: Any,
    out_dir: str | Path,
    *,
    name: str,
    dataset: str,
    metrics: dict[str, float],
    notes: str = "",
    decimals: int = 6,
) -> dict[str, Any]:
    params = vectorizer.get_params()
    unsupported = {
        "analyzer": params["analyzer"] != "word",
        "tokenizer": params["tokenizer"] is not None,
        "preprocessor": params["preprocessor"] is not None,
        "stop_words": params["stop_words"] is not None,
        "strip_accents": params["strip_accents"] is not None,
        "norm": params["norm"] not in ("l2", None),
        "use_idf": not params["use_idf"],
        "binary": params["binary"],
    }
    if any(unsupported.values()):
        raise ValueError(
            f"unsupported vectorizer settings: {[k for k, v in unsupported.items() if v]}"
        )
    vocabulary = {term: int(index) for term, index in vectorizer.vocabulary_.items()}
    payload = {
        "format": FORMAT,
        "lowercase": bool(params["lowercase"]),
        "token_pattern": params["token_pattern"],
        "ngram_range": list(params["ngram_range"]),
        "sublinear_tf": bool(params["sublinear_tf"]),
        "norm": params["norm"],
        "vocabulary": vocabulary,
        "idf": [round(float(v), decimals) for v in vectorizer.idf_],
        "classes": [str(c) for c in model.classes_],
        "coef": [[round(float(v), decimals) for v in row] for row in model.coef_],
        "intercept": [round(float(v), decimals) for v in model.intercept_],
    }
    import sklearn

    card = {
        "name": name,
        "version": datetime.date.today().strftime("%Y.%m.%d"),
        "trained_at": datetime.datetime.now(datetime.UTC).isoformat(),
        "framework": "scikit-learn",
        "framework_version": sklearn.__version__,
        "dataset": dataset,
        "features": [FORMAT],
        "metrics": {k: float(v) for k, v in metrics.items()},
        "notes": notes,
    }
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "model.json").write_text(json.dumps(payload, separators=(",", ":")), encoding="utf-8")
    (out / "model_card.json").write_text(json.dumps(card, indent=2), encoding="utf-8")
    return card


card = export_tfidf_logreg(
    vectorizer, model, "/kaggle/working", name="review_sentiment",
    dataset="Trip Advisor Hotel Reviews (Kaggle andrewmvd/trip-advisor-hotel-reviews)",
    metrics=metrics, notes="1-2 stars negative, 3 neutral, 4-5 positive; 80/20 stratified split, seed 7.",
)
print(card)